# Volcano Plot visualizations

### Resources
https://en.wikipedia.org/wiki/Volcano_plot_(statistics)  
https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.levene.html   
https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.kruskal.html   
https://dash.plotly.com/dash-bio/volcanoplot  

In [1]:
%cd ../..

/home/bhkuser/bhklab/katy/readii_2_roqc


In [29]:
from damply import dirs
import pandas as pd 
from scipy.stats import levene, kruskal
import numpy as np
import plotly.graph_objects as go
import dash_bio
import kaleido
from readii.process.subset import getOnlyPyradiomicsFeatures, dropUpToFeature
from readii_2_roqc.utils.loaders import load_signature_config

In [ ]:
# df = pd.read_csv('https://raw.githubusercontent.com/plotly/dash-bio-docs-files/master/volcano_data1.csv')

In [50]:
dataset = "TCIA_HEAD-NECK-RADIOMICS-HN1_windowed"
nc = "shuffled_non_roi"

In [51]:
# read in the features for original vs. randomized roi
o_data = pd.read_csv(dirs.RESULTS / dataset / "features/pyradiomics/original_512_512_n/linear_all_images_features/original_full_features.csv", index_col=0)
nc_data = pd.read_csv(dirs.RESULTS / dataset / "features/pyradiomics/original_512_512_n/linear_all_images_features" / f"{nc}_features.csv", index_col=0)

if dataset == "TCIA_RADCURE_windowed":
    test_samples = pd.read_csv(dirs.PROCDATA / f"{dataset}/test_samples.csv")
    o_data = o_data.loc[test_samples.SampleID]
    nc_data = nc_data.loc[test_samples.SampleID]

# get just radiomics features
o_features = getOnlyPyradiomicsFeatures(o_data)
nc_features = getOnlyPyradiomicsFeatures(nc_data)

o_features = dropUpToFeature(o_features, 'original_firstorder_10Percentile', True)
nc_features = dropUpToFeature(nc_features, 'original_firstorder_10Percentile', True)

# calculate the mean value for each feature
o_means = o_features.mean(axis=0)
nc_means = nc_features.mean(axis=0)

# Calculate log2 fold change between each of the features
# Add small epsilon value to handle divide by zero errors
epsilon = 0
# Make means absolute so log fold change can be calculated (can't handle negative values)
log2_fold_change = np.log2(np.abs(o_means) + epsilon / np.abs(nc_means) + epsilon)

# do the levene test
levene_stat, levene_p = levene(o_features.to_numpy(), nc_features.to_numpy(), axis=0)
levene_p = pd.Series(levene_p, index=o_features.columns)

# Check if any of the values are significant
if np.any(levene_p > 0.05):
    print("Populations do not have equal variances. ANOVA assumption violated, perform Kruskal Wallis")

    # tests the null hypothesis that the population median of all of the groups are equal
    kruskal_stat, kruskal_p = kruskal(o_features.to_numpy(), nc_features.to_numpy(), axis=0)
    log_kruskal_p = -np.log10(kruskal_p)
    kruskal_p = pd.Series(kruskal_p, index=o_features.columns)
    log_kruskal_p = pd.Series(log_kruskal_p, index=o_features.columns)

    volcano_df = pd.DataFrame([kruskal_p, log2_fold_change], index=['P', 'EFFECTSIZE']).T

else:
    print("Populations have equal variance. Proceed with ANOVA.")

# Move the features from the index to their own column
volcano_df = volcano_df.reset_index(names='GENE')

volcano_fig = dash_bio.VolcanoPlot(
    dataframe = volcano_df,
    snp = 'GENE',
    logp = True
)

volcano_fig.update_layout(
    title=f'Volcano Plot - original vs. {nc}',
    title_subtitle=dict(text=dataset),
    xaxis_title="Log2 Fold Change", 
    yaxis_title="-Log10 P-value",
    width = 850,
    height=500
    )

volcano_fig.show()

Populations do not have equal variances. ANOVA assumption violated, perform Kruskal Wallis


## Plot by image filter 

In [52]:
# Filter volcano data by image filter
image_filter = 'original'

filtered_vol_df = volcano_df.loc[volcano_df['GENE'].str.contains(image_filter)]
filtered_vol_df.reset_index(inplace=True)

filtered_volcano_fig = dash_bio.VolcanoPlot(
    dataframe = filtered_vol_df,
    snp = 'GENE',
    logp = True
)

filtered_volcano_fig.update_layout(
    title=f'Volcano Plot - original vs. {nc} - {image_filter} filter',
    title_subtitle=dict(text=dataset),
    xaxis_title="Log2 Fold Change", 
    yaxis_title="-Log10 P-value",
    width = 800,
    height= 500
    )

filtered_volcano_fig.show()

## Plot by signature features

In [54]:
signature_name = "choi_opc_hpv_2020"
signature = load_signature_config(signature_name)

sig_feats = signature.index.to_list()

signature_vol_df = volcano_df.loc[volcano_df.GENE.isin(sig_feats)]
signature_vol_df.reset_index(inplace=True)

signature_volcano_fig = dash_bio.VolcanoPlot(
    dataframe = signature_vol_df,
    snp = 'GENE',
    logp = True
)

signature_volcano_fig.update_layout(
    title=f'Volcano Plot - original vs. {nc} - {signature_name}',
    title_subtitle=dict(text=dataset),
    xaxis_title="Log2 Fold Change", 
    yaxis_title="-Log10 P-value",
    width = 800,
    height= 500
    )

signature_volcano_fig.show()